## Imports

In [73]:
# Maths & Operations related imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import gc
import random
import shutil

# Pytorch related imports
import torch
from PIL import Image

# Sklearn related imports
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

# PM4Py related imports
import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from pm4py.visualization.bpmn import visualizer as bpmn_visualizer
from pm4py.objects.bpmn.exporter import exporter as bpmn_exporter
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness_evaluator
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator
from pm4py.visualization.dfg import visualizer as dfg_visualizer
from pm4py.algo.discovery.correlation_mining import algorithm as correlation_miner

#Transformers related imports
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

## CSV Reader

In [74]:
def read_ui_log_as_dataframe(log_path):
  return pd.read_csv(log_path, sep=";")

## Feature Extraction

In [75]:
def extract_features_from_images(df, image_col, text_col, image_weight, text_weight, img_dir):
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    combined_features = []

    for _, row in df.iterrows():
        text = row[text_col]
        image_path = os.path.join(img_dir, row[image_col])
        
        if not os.path.exists(image_path):
            raise ValueError(f"La imagen no existe en {image_path}")

        image = Image.open(image_path)
        inputs = processor(text=[text], images=image, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs)

        image_features = outputs.image_embeds.cpu().numpy().flatten() * image_weight
        text_features = outputs.text_embeds.cpu().numpy().flatten() * text_weight
        
        combined_feature = np.hstack((image_features, text_features))
        combined_features.append(combined_feature)

    df['combined_features'] = combined_features

    return df

In [4]:
def extract_features_from_images_with_tokenizer(df, image_col, text_col, image_weight, text_weight, img_dir, header_txt=False, text_path_col="header_txt"):
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

    combined_features = []

    for _, row in df.iterrows():
        if header_txt:
            txt_path = os.path.join("logs/invoice_def", "ocr_results", row[text_path_col])
            if not os.path.exists(txt_path):
                raise FileNotFoundError(f"El archivo de texto no existe: {txt_path}")
            with open(txt_path, 'r') as file:
                text = file.read()
        else:
            text = row[text_col]

        # Tokenizar el texto
        input_ids = tokenizer(text, return_tensors="pt", truncation=True)

        # Cargar la imagen
        image_path = os.path.join(img_dir, row[image_col])
        if not os.path.exists(image_path):
            raise ValueError(f"La imagen no existe en {image_path}")

        # Abre la imagen y la procesa con el modelo CLIP
        image = Image.open(image_path)
        image_inputs = processor(images=[image], return_tensors="pt")

        # Combina las entradas de texto e imagen y pasarlo al modelo
        inputs = {'input_ids': input_ids['input_ids'], 'attention_mask': input_ids['attention_mask'], 'pixel_values': image_inputs['pixel_values']}

        with torch.no_grad():
            outputs = model(**inputs)

        image_features = outputs.image_embeds.cpu().numpy().flatten() * image_weight
        text_features = outputs.text_embeds.cpu().numpy().flatten() * text_weight
        combined_feature = np.hstack((image_features, text_features))
        combined_features.append(combined_feature)

    df['combined_features'] = combined_features
    return df


## Clustering

In [76]:
def cluster_images(df, n_clusters_range, use_pca, n_components):
    features = np.array(df['combined_features'].tolist())
    
    if use_pca:
        pca = PCA(n_components=n_components)
        features = pca.fit_transform(features)
        print(f"PCA aplicado: {features.shape[1]} componentes retenidos")

    clustering_scores = {
        'n_clusters': [],
        'silhouette_score': [],
        'davies_bouldin_score': [],
        'calinski_harabasz_score': []
    }

    for k in range(*n_clusters_range):
        clustering = AgglomerativeClustering(n_clusters=k).fit(features)
        labels = clustering.labels_

        clustering_scores['n_clusters'].append(k)
        clustering_scores['silhouette_score'].append(silhouette_score(features, labels))
        clustering_scores['davies_bouldin_score'].append(davies_bouldin_score(features, labels))
        clustering_scores['calinski_harabasz_score'].append(calinski_harabasz_score(features, labels))

    # Encuentra el índice del número óptimo de clústeres basado en la mejor puntuación Silhouette
    optimal_index = np.argmax(clustering_scores['silhouette_score'])
    optimal_clusters = clustering_scores['n_clusters'][optimal_index]

    # Ejecutar el clustering con el número óptimo de clústeres
    best_clustering = AgglomerativeClustering(n_clusters=optimal_clusters).fit(features)
    df['activity_label'] = best_clustering.labels_

    # Obtener las métricas para el número óptimo de clústeres
    optimal_metrics = {
        'silhouette_score': clustering_scores['silhouette_score'][optimal_index],
        'davies_bouldin_score': clustering_scores['davies_bouldin_score'][optimal_index],
        'calinski_harabasz_score': clustering_scores['calinski_harabasz_score'][optimal_index]
    }

    return df, clustering_scores, optimal_clusters, optimal_metrics


## Case ID Allocation

In [77]:
def auto_process_id_assignment(df):
    activity_inicial = df['activity_label'].iloc[0]
    process_id = 1
    process_ids = [process_id]  
    for index, row in df.iterrows():
        if index != 0:  
            if row['activity_label'] == activity_inicial:
                process_id += 1
            process_ids.append(process_id)
        else:
            continue
    df['process_id'] = process_ids
    return df


def eliminar_acciones_duplicadas(df, columna_label='activity_label'):
    while True:
        mascaras_para_eliminar = df[columna_label].eq(df[columna_label].shift())
        if mascaras_para_eliminar.sum() == 0:
            break
        df = df[~mascaras_para_eliminar].reset_index(drop=True)
    return df

## BPMN / Petrinet Generator

In [56]:
def calculate_metrics_petri_net(event_log, net, initial_marking, final_marking):
    fitness_result = pm4py.fitness_token_based_replay(event_log, net, initial_marking, final_marking)
    fitness = fitness_result['average_trace_fitness']
    precision = pm4py.precision_token_based_replay(event_log, net, initial_marking, final_marking)
    simplicity = simplicity_evaluator.apply(net)
    generalization = generalization_evaluator.apply(event_log, net, initial_marking, final_marking)
    return fitness, precision, simplicity, generalization

In [70]:
def format_and_convert_to_event_log(df, timestamp_col):
    formatted_df = pm4py.format_dataframe(df, case_id='process_id', activity_key='activity_label', timestamp_key=timestamp_col)
    return pm4py.convert_to_event_log(formatted_df)

def save_dot_file(dot, filename):
    dot_path = os.path.join('results', filename)
    with open(dot_path, 'w') as f:
        f.write(dot.source)

def save_bpmn_model(bpmn_model, filename):
    bpmn_exporter.apply(bpmn_model, os.path.join('results', filename))

def calculate_and_return_metrics(event_log, net, initial_marking, final_marking):
    fitness, precision, simplicity, generalization = calculate_metrics_petri_net(event_log, net, initial_marking, final_marking)
    return fitness, precision, simplicity, generalization

def bpmn_process_inductive_miner(df, timestamp_col):
    event_log = format_and_convert_to_event_log(df, timestamp_col)
    bpmn_model = pm4py.discover_bpmn_inductive(event_log)
    dot = bpmn_visualizer.apply(bpmn_model)
    save_dot_file(dot, 'bpmn_inductive.dot')
    save_bpmn_model(bpmn_model, 'bpmn_inductive.bpmn')

def bpmn_process_alpha_miner(df, timestamp_col):
    event_log = format_and_convert_to_event_log(df, timestamp_col)
    net, initial_marking, final_marking = pm4py.discover_petri_net_alpha(event_log)
    bpmn_model = pm4py.convert_to_bpmn(net, initial_marking, final_marking)
    dot = bpmn_visualizer.apply(bpmn_model)
    save_dot_file(dot, 'bpmn_alpha.dot')
    save_bpmn_model(bpmn_model, 'bpmn_alpha.bpmn')
    return calculate_and_return_metrics(event_log, net, initial_marking, final_marking)

def bpmn_process_heuristic_miner(df, timestamp_col):
    event_log = format_and_convert_to_event_log(df, timestamp_col)
    heu_net = pm4py.discover_heuristics_net(event_log, dependency_threshold=0.99)
    net, initial_marking, final_marking = pm4py.convert_to_petri_net(heu_net)
    bpmn_model = pm4py.convert_to_bpmn(net, initial_marking, final_marking)
    dot = bpmn_visualizer.apply(bpmn_model)
    save_dot_file(dot, 'bpmn_heuristic.dot')
    save_bpmn_model(bpmn_model, 'bpmn_heuristic.bpmn')
    return calculate_and_return_metrics(event_log, net, initial_marking, final_marking)

def bpmn_process_ilp_miner(df, timestamp_col):
    event_log = format_and_convert_to_event_log(df, timestamp_col)
    net, initial_marking, final_marking = pm4py.discover_petri_net_ilp(event_log)
    bpmn_model = pm4py.convert_to_bpmn(net, initial_marking, final_marking)
    dot = bpmn_visualizer.apply(bpmn_model)
    save_dot_file(dot, 'bpmn_ilp.dot')
    save_bpmn_model(bpmn_model, 'bpmn_ilp.bpmn')
    return calculate_and_return_metrics(event_log, net, initial_marking, final_marking)

def petri_net_process(df, timestamp_col):
    event_log = format_and_convert_to_event_log(df, timestamp_col)
    process_tree = inductive_miner.apply(event_log)
    net, initial_marking, final_marking = pm4py.convert_to_petri_net(process_tree)
    dot = pn_visualizer.apply(net, initial_marking, final_marking)
    save_dot_file(dot, 'pn.dot')
    return calculate_and_return_metrics(event_log, net, initial_marking, final_marking)

## Auxiliary Functions

In [78]:
def overwrite_csv(df, file_path):
    """Escribe un DataFrame a un archivo CSV, sobrescribiendo el archivo existente."""
    try:
        if os.path.exists(file_path):
            os.remove(file_path)
        df.to_csv(file_path, index=False)
    except Exception as e:
        print(f"Error al escribir el archivo CSV: {e}")

def move_and_overwrite(source, destination):
    """Mueve un archivo de una ubicación a otra y lo sobrescribe si ya existe."""
    if os.path.exists(destination):
        os.remove(destination)
    shutil.move(source, destination)
    
def clear_caches():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_fresh_data():
    return read_ui_log_as_dataframe(log_path)

## Case selection

In [50]:
#invoice_management
log_path = 'logs/invoice_def/log.csv'
image_col = 'screenshot'
image_dir = 'resources/#OLD/invoice_def'
text_col = 'header'
timestamp_col = 'timestamp'

In [17]:
#invoice_management (customer view patch)
log_path = 'logs/invoice_def/log.csv'
image_col = 'screenshot'
image_dir = 'resources/#OLD/invoice_def_customer_view'
text_col = 'header'
timestamp_col = 'timestamp'

In [23]:
#payment notification
log_path = 'logs/SC50_Rebuild/log.csv'
image_col = 'screenshot'
image_dir = 'resources/#OLD/SC50_Rebuild'
text_col = 'header'
timestamp_col = 'timestamp'

In [26]:
#payment notification - dual monitoring
log_path = 'logs/SC50_Rebuild/log.csv'
image_col = 'screenshot'
image_dir = 'resources/#OLD/SC50_Hybrid'
text_col = 'header'
timestamp_col = 'timestamp'

## Execute and save run

In [82]:
model = 'clip'
n_clusters_range = (2, 11)
n_components = 0.95
use_pca = False
tokeniza = False #¿Tokenizamos?
header_txt = False #¿Usamos el texto completo?

# Directorio principal para guardar los casos de estudio
case_study_name = "sample2"  # Cambiar el nombre del caso de estudio
root_dir = os.path.join("executions", case_study_name)
os.makedirs(root_dir, exist_ok=True)

# Arrays donde se almacenarán resultados
results = []
metrics_inductive = []
metrics_alpha = []
metrics_heuristic = []
metrics_ilp = []
metrics_dfg = []

# Información de las ejecuciones a realizar
executions = [
    {'exec': 1, 'image_weight': 1, 'text_weight': 0},
    {'exec': 2, 'image_weight': 0.8, 'text_weight': 0.2},
    {'exec': 3, 'image_weight': 0.6, 'text_weight': 0.4},
    {'exec': 4, 'image_weight': 0.5, 'text_weight': 0.5},
    {'exec': 5, 'image_weight': 0.4, 'text_weight': 0.6},
    {'exec': 6, 'image_weight': 0.2, 'text_weight': 0.8},
    {'exec': 7, 'image_weight': 0, 'text_weight': 1}
]

In [83]:
# Función para leer y procesar el log de UI
def read_and_process_log(log_path):
    df = read_ui_log_as_dataframe(log_path)
    clear_caches()
    return df

# Función para configurar la semilla aleatoria
def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Función para crear el directorio de ejecución
def create_execution_directory(case_study_name, exec, root_dir):
    exec_dir = f"{case_study_name}_{exec['image_weight']}_{exec['text_weight']}"
    exec_path = os.path.join(root_dir, exec_dir)
    os.makedirs(exec_path, exist_ok=True)
    return exec_path

# Función para extraer características de las imágenes
def extract_features(df, exec, image_col, text_col, image_weight, text_weight, image_dir, header_txt, tokeniza):
    if tokeniza:
        return extract_features_from_images_with_tokenizer(df, image_col, text_col, image_weight, text_weight, image_dir, header_txt, text_path_col='header_txt')
    else:
        return extract_features_from_images(df, image_col, text_col, image_weight, text_weight, image_dir)

# Función para mover y sobrescribir archivos
def move_and_overwrite(source, destination):
    if os.path.exists(source):
        if os.path.exists(destination):
            os.remove(destination)
        shutil.move(source, destination)

# Función para procesar el clustering y las métricas
def process_clustering_and_metrics(df, n_clusters_range, use_pca, n_components):
    df, clustering_scores, optimal_clusters, optimal_metrics = cluster_images(df, n_clusters_range, use_pca, n_components)
    df = auto_process_id_assignment(df)
    df = eliminar_acciones_duplicadas(df, columna_label='activity_label')
    return df, optimal_metrics

# Función para guardar métricas en el archivo de resultados
def save_metrics(result_entry, results):
    results.append(result_entry)

# Función para procesar usando diferentes miners
def process_with_miners(df, timestamp_col):
    fitness_inductive, precision_inductive, simplicity_inductive, generalization_inductive = petri_net_process(df, timestamp_col)
    bpmn_process_inductive_miner(df, timestamp_col)
    fitness_alpha, precision_alpha, simplicity_alpha, generalization_alpha = bpmn_process_alpha_miner(df, timestamp_col)
    fitness_heuristic, precision_heuristic, simplicity_heuristic, generalization_heuristic = bpmn_process_heuristic_miner(df, timestamp_col)
    fitness_ilp, precision_ilp, simplicity_ilp, generalization_ilp = bpmn_process_ilp_miner(df, timestamp_col)
    
    return {
        'Fitness_Inductive': fitness_inductive,
        'Precision_Inductive': precision_inductive,
        'Simplicity_Inductive': simplicity_inductive,
        'Generalization_Inductive': generalization_inductive,
        'Fitness_Alpha': fitness_alpha,
        'Precision_Alpha': precision_alpha,
        'Simplicity_Alpha': simplicity_alpha,
        'Generalization_Alpha': generalization_alpha,
        'Fitness_Heuristic': fitness_heuristic,
        'Precision_Heuristic': precision_heuristic,
        'Simplicity_Heuristic': simplicity_heuristic,
        'Generalization_Heuristic': generalization_heuristic,
        'Fitness_ILP': fitness_ilp,
        'Precision_ILP': precision_ilp,
        'Simplicity_ILP': simplicity_ilp,
        'Generalization_ILP': generalization_ilp,
    }

# Función para mover y guardar archivos de resultados
def move_results_files(exec_path):
    move_and_overwrite('results/bpmn_inductive.dot', os.path.join(exec_path, 'bpmn_inductive.dot'))
    move_and_overwrite('results/bpmn_inductive.bpmn', os.path.join(exec_path, 'bpmn_inductive.bpmn'))
    move_and_overwrite('results/bpmn_alpha.dot', os.path.join(exec_path, 'bpmn_alpha.dot'))
    move_and_overwrite('results/bpmn_alpha.bpmn', os.path.join(exec_path, 'bpmn_alpha.bpmn'))
    move_and_overwrite('results/bpmn_heuristic.dot', os.path.join(exec_path, 'bpmn_heuristic.dot'))
    move_and_overwrite('results/bpmn_heuristic.bpmn', os.path.join(exec_path, 'bpmn_heuristic.bpmn'))
    move_and_overwrite('results/bpmn_ilp.dot', os.path.join(exec_path, 'bpmn_ilp.dot'))
    move_and_overwrite('results/bpmn_ilp.bpmn', os.path.join(exec_path, 'bpmn_ilp.bpmn'))

In [84]:
for exec in executions:

    df = read_and_process_log(log_path)
    set_random_seed(42)
    
    exec_path = create_execution_directory(case_study_name, exec, root_dir)
    
    image_weight = exec['image_weight']
    text_weight = exec['text_weight']
    
    df = extract_features(df, exec, image_col, text_col, image_weight, text_weight, image_dir, header_txt, tokeniza)
    
    df, optimal_metrics = process_clustering_and_metrics(df, n_clusters_range, use_pca, n_components)
    
    metrics = process_with_miners(df, timestamp_col)
    
    df.to_csv(os.path.join(exec_path, 'df.csv'), index=False)
    
    move_results_files(exec_path)

    result_entry = {
        'exec': exec['exec'],
        'image_weight': image_weight,
        'text_weight': text_weight,
        'Silhouette': optimal_metrics['silhouette_score'],
        'Davies-Bouldin': optimal_metrics['davies_bouldin_score'],
        'Calinski-Harabasz': optimal_metrics['calinski_harabasz_score'],
        **metrics
    }
    
    save_metrics(result_entry, results)

results_df = pd.DataFrame(results)
overwrite_csv(results_df, os.path.join(root_dir, 'resultados.csv'))

replaying log with TBR, completed traces ::   0%|          | 0/4 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/4 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/4 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

...

replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

discovering Petri net using ILP miner, completed causal relations ::   0%|          | 0/10 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

## Testing

In [ ]:
data = {
   'activity_label': [
       7, 6, 2, 0, 0, 1, 3, 7, 6, 2, 0, 0, 1, 3, 7, 6, 5, 8, 7, 6, 2, 0, 0, 1, 4, 
       7, 6, 5, 8, 7, 6, 2, 0, 0, 1, 3, 7, 6, 5, 9, 7, 6, 2, 0, 0, 1, 3, 7, 6, 5, 
       9, 7, 6, 2, 0, 0, 1, 4
   ],
   'timestamp': [
       '10:00:29', '10:00:31', '10:00:33', '10:00:35', '10:00:37', '10:00:39', '10:00:41', 
       '10:00:43', '10:00:45', '10:00:47', '10:00:49', '10:00:51', '10:00:53', '10:00:55', 
       '10:00:57', '10:00:59', '10:01:01', '10:01:03', '10:01:05', '10:01:07', '10:01:09', 
       '10:01:11', '10:01:13', '10:01:15', '10:01:17', '10:01:19', '10:01:21', '10:01:23', 
       '10:01:25', '10:01:27', '10:01:29', '10:01:31', '10:01:33', '10:01:35', '10:01:37', 
       '10:01:39', '10:01:41', '10:01:43', '10:01:45', '10:01:47', '10:01:49', '10:01:51', 
       '10:01:53', '10:01:55', '10:01:57', '10:01:59', '10:02:01', '10:02:03', '10:02:05', 
       '10:02:07', '10:02:09', '10:02:11', '10:02:13', '10:02:15', '10:02:17', '10:02:19', 
       '10:02:23', '10:02:30'
   ]
}

df = pd.DataFrame(data)
df = auto_process_id_assignment(df)
df = eliminar_acciones_duplicadas(df, columna_label='activity_label')
fitness, precision, simplicity, generalization = bpmn_process_dfg_miner(df, 'timestamp')